In [1]:
# Parameters
DB_PATH          = "../../../DB/oedb_baseline_v3.db"
BENCHMARK_PATH   = "../../../data/input_data/benchmark_trainingset.xlsx"
BENCHMARK_SHEET  = "participants"
MATCHED_CSV_PATH = "matched_participants.csv"
NOTEGROUP_ID_MIN = 1
NOTEGROUP_ID_MAX = 23

EVAL_FIELDS = [
    "full_name", "session_identifier", "gender", "projectID_age",
    "learning_route", "participant_group", "status", "place_of_origin", "language_group",
    "first_arrival_date", "municipality", "other_information"
]

DATE_FIELDS = {"first_arrival_date"}

# Thresholds per the value-agreeing rule table
THRESHOLDS = {
    "full_name":           0.85,   # token sort ratio
    "session_identifier":  0.30,   # token sort ratio, not strict
    "gender":               0.90,  # semantic similarity
    "learning_route":       0.90,  # strip + ratio
    "participant_group":    0.90,  # semantic similarity
    "place_of_origin":      0.85,  # semantic similarity
    "municipality":         0.85,  # ratio
    "other_information_key":   0.50,  # semantic similarity on keys
    "other_information_value": 0.85,  # ratio on values
}

In [2]:
import sqlite3
import json
import re
import pandas as pd
from datetime import datetime
from rapidfuzz import fuzz
from sentence_transformers import SentenceTransformer, util

# multilingual model — supports Dutch, English, Arabic, and 50+ other languages
model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")

def load_etl(db_path, id_min, id_max):
    con = sqlite3.connect(db_path)
    df = pd.read_sql_query(
        """SELECT participantID, notegroupID, full_name, session_identifier,
                  gender, projectID_age, learning_route, participant_group,
                  place_of_origin, language_group, first_arrival_date,
                  municipality, other_information
           FROM participants
           WHERE notegroupID BETWEEN ? AND ?""",
        con, params=(id_min, id_max)
    )
    con.close()
    df["participantID"] = df["participantID"].astype(int)
    return df.set_index("participantID")

def load_benchmark(xlsx_path, sheet, id_min, id_max):
    df = pd.read_excel(xlsx_path, sheet_name=sheet, dtype=str)
    df["notegroupID"]  = df["notegroupID"].astype(int)
    df["participantID"] = df["participantID"].astype(int)
    df = df[df["notegroupID"].between(id_min, id_max)]
    return df.set_index("participantID")

def load_matched(csv_path):
    df = pd.read_csv(csv_path)
    df["etl_participantID"] = df["etl_participantID"].astype(int)
    df["bm_participantID"]  = df["bm_participantID"].astype(int)
    return df

etl     = load_etl(DB_PATH, NOTEGROUP_ID_MIN, NOTEGROUP_ID_MAX)
bm      = load_benchmark(BENCHMARK_PATH, BENCHMARK_SHEET, NOTEGROUP_ID_MIN, NOTEGROUP_ID_MAX)
matched = load_matched(MATCHED_CSV_PATH)

matched_etl = set(matched["etl_participantID"])
matched_bm  = set(matched["bm_participantID"])
etl_only    = sorted(set(etl.index) - matched_etl)
bm_only     = sorted(set(bm.index)  - matched_bm)

print(f"ETL records   : {len(etl)}")
print(f"BM records    : {len(bm)}")
print(f"Matched pairs : {len(matched)}")
print(f"ETL-only (FP) : {len(etl_only)}")
print(f"BM-only  (FN) : {len(bm_only)}")

/Users/weiyizzz/PycharmProjects/OE_ETL/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 199/199 [00:00<00:00, 11071.46it/s]


ETL records   : 93
BM records    : 93
Matched pairs : 93
ETL-only (FP) : 0
BM-only  (FN) : 0


In [3]:
def extract_age(projectID_age_str):
    if pd.isna(projectID_age_str) or not projectID_age_str:
        return None
    try:
        d = json.loads(projectID_age_str)
        return next(iter(d.values()))
    except (json.JSONDecodeError, StopIteration, TypeError):
        return None

def normalise_eval(val, is_date=False):
    if pd.isna(val) or str(val).strip() in ("", "None", "nan", "NaT", "{}", "[]"):
        return None
    s = str(val).strip()
    if is_date:
        s = s.split(" ")[0]
        for fmt in ("%Y-%m-%d", "%m/%d/%Y", "%d-%m-%Y", "%Y/%m/%d"):
            try:
                return datetime.strptime(s, fmt).strftime("%Y-%m-%d")
            except ValueError:
                continue
        return s.lower()
    return s.lower()

def semantic_similarity(a, b):
    """Cosine similarity between two strings using sentence embeddings, scaled 0-1."""
    if a is None or b is None:
        return None
    emb = model.encode([a, b], convert_to_tensor=True)
    return util.cos_sim(emb[0], emb[1]).item()

def compute_counts(etl_val, bm_val, is_date=False, sim_threshold=None,
                    semantic_threshold=None):
    """
    Return (TP, FP, FN, TN) for one field comparison.
    sim_threshold        : use fuzz.ratio / 100 >= threshold
    semantic_threshold    : use sentence-embedding cosine similarity >= threshold
    Neither set           : exact string equality (default)
    """
    e = normalise_eval(etl_val, is_date)
    b = normalise_eval(bm_val,  is_date)
    if e is not None and b is not None:
        if semantic_threshold is not None:
            match = semantic_similarity(e, b) >= semantic_threshold
        elif sim_threshold is not None:
            match = (fuzz.ratio(e, b) / 100) >= sim_threshold
        else:
            match = (e == b)
        return (1, 0, 0, 0) if match else (0, 1, 1, 0)
    if e is not None and b is None:
        return (0, 1, 0, 0)   # hallucinated → FP
    if e is None and b is not None:
        return (0, 0, 1, 0)   # missed → FN
    return (0, 0, 0, 1)       # both null → TN

def normalise_learning_route(val):
    s = normalise_eval(val)
    if s is None:
        return None
    s = re.sub(r'\b(route|routes)\b', '', s)
    s = re.sub(r'[-,\s]+', '', s)
    return s.strip() or None

def normalise_language_group(val):
    """Parse language_group into a set (order-independent), or None if empty."""
    if pd.isna(val) or str(val).strip() in ("", "None", "nan", "[]"):
        return None
    s = str(val).strip().strip("[]")
    items = re.split(r'[,;]', s)
    result = {item.strip().strip("'\"").lower() for item in items if item.strip()}
    return result if result else None

def parse_other_information(val):
    """Parse other_information into a dict, or None if empty/invalid."""
    if pd.isna(val) or str(val).strip() in ("", "None", "nan", "{}"):
        return None
    try:
        d = json.loads(val)
        return d if isinstance(d, dict) and d else None
    except (json.JSONDecodeError, TypeError):
        return None

def other_information_match(etl_val, bm_val,
                              key_threshold=THRESHOLDS["other_information_key"],
                              value_threshold=THRESHOLDS["other_information_value"]):
    """
    Compare two other_information dicts, tolerant of key ordering.
    A pair (etl_key, etl_val) matches (bm_key, bm_val) if:
      - semantic_similarity(etl_key, bm_key) >= key_threshold
      - fuzz.ratio(etl_val, bm_val) / 100 >= value_threshold
    Whole-field decision: match only if every pair on both sides is matched
    with no leftovers.
    """
    e = parse_other_information(etl_val)
    b = parse_other_information(bm_val)

    if e is None and b is None:
        return (0, 0, 0, 1)   # both null → TN
    if e is None:
        return (0, 0, 1, 0)   # missed → FN
    if b is None:
        return (0, 1, 0, 0)   # hallucinated → FP

    used_etl_keys = set()
    all_bm_matched = True
    for bk, bv in b.items():
        found = False
        for ek, ev in e.items():
            if ek in used_etl_keys:
                continue
            key_sim = semantic_similarity(normalise_eval(ek), normalise_eval(bk))
            val_sim = fuzz.ratio(normalise_eval(str(ev)), normalise_eval(str(bv))) / 100
            if key_sim >= key_threshold and val_sim >= value_threshold:
                used_etl_keys.add(ek)
                found = True
                break
        if not found:
            all_bm_matched = False
            break

    is_match = all_bm_matched and len(used_etl_keys) == len(e) == len(b)
    return (1, 0, 0, 0) if is_match else (0, 1, 1, 0)

def safe_div(num, den):
    return round(num / den, 4) if den > 0 else None

def metrics_from_counts(TP, FP, FN, TN):
    accuracy  = safe_div(TP + TN, TP + FP + FN + TN)
    precision = safe_div(TP, TP + FP)
    recall    = safe_div(TP, TP + FN)
    f1 = round(2 * precision * recall / (precision + recall), 4) \
         if precision and recall and (precision + recall) > 0 else None
    return dict(TP=TP, FP=FP, FN=FN, TN=TN,
                accuracy=accuracy, precision=precision, recall=recall, F1=f1)

In [4]:
totals = {f: dict(TP=0, FP=0, FN=0, TN=0) for f in EVAL_FIELDS}

for _, row in matched.iterrows():
    ei = row["etl_participantID"]
    bi = row["bm_participantID"]
    for field in EVAL_FIELDS:
        etl_val = etl.at[ei, field] if field in etl.columns else None
        bm_val  = bm.at[bi, field]  if field in bm.columns  else None

        if field == "projectID_age":
            etl_val = extract_age(etl_val)
            bm_val  = extract_age(bm_val)
            tp, fp, fn, tn = compute_counts(etl_val, bm_val)

        elif field == "full_name":
            tp, fp, fn, tn = compute_counts(etl_val, bm_val, sim_threshold=THRESHOLDS["full_name"])

        elif field == "session_identifier":
            tp, fp, fn, tn = compute_counts(etl_val, bm_val, sim_threshold=THRESHOLDS["session_identifier"])

        elif field == "gender":
            tp, fp, fn, tn = compute_counts(etl_val, bm_val, semantic_threshold=THRESHOLDS["gender"])

        elif field == "learning_route":
            etl_val = normalise_learning_route(etl_val)
            bm_val  = normalise_learning_route(bm_val)
            tp, fp, fn, tn = compute_counts(etl_val, bm_val, sim_threshold=THRESHOLDS["learning_route"])

        elif field == "participant_group":
            tp, fp, fn, tn = compute_counts(etl_val, bm_val, semantic_threshold=THRESHOLDS["participant_group"])

        elif field == "place_of_origin":
            tp, fp, fn, tn = compute_counts(etl_val, bm_val, semantic_threshold=THRESHOLDS["place_of_origin"])

        elif field == "language_group":
            e_set = normalise_language_group(etl_val)
            b_set = normalise_language_group(bm_val)
            if e_set is not None and b_set is not None:
                tp, fp, fn, tn = (1, 0, 0, 0) if e_set == b_set else (0, 1, 1, 0)
            elif e_set is not None and b_set is None:
                tp, fp, fn, tn = (0, 1, 0, 0)
            elif e_set is None and b_set is not None:
                tp, fp, fn, tn = (0, 0, 1, 0)
            else:
                tp, fp, fn, tn = (0, 0, 0, 1)

        elif field == "municipality":
            tp, fp, fn, tn = compute_counts(etl_val, bm_val, sim_threshold=THRESHOLDS["municipality"])

        elif field == "other_information":
            tp, fp, fn, tn = other_information_match(etl_val, bm_val)

        else:
            tp, fp, fn, tn = compute_counts(etl_val, bm_val, is_date=(field in DATE_FIELDS))

        totals[field]["TP"] += tp; totals[field]["FP"] += fp
        totals[field]["FN"] += fn; totals[field]["TN"] += tn

# ETL-only rows → every field counts as FP
for ei in etl_only:
    for field in EVAL_FIELDS:
        totals[field]["FP"] += 1

# BM-only rows → every field counts as FN
for bi in bm_only:
    for field in EVAL_FIELDS:
        totals[field]["FN"] += 1

In [5]:
rows = []
for field in EVAL_FIELDS:
    m = metrics_from_counts(**totals[field])
    rows.append({"field": field, **m})

overall = {k: sum(totals[f][k] for f in EVAL_FIELDS) for k in ("TP","FP","FN","TN")}
m_all = metrics_from_counts(**overall)
rows.append({"field": "OVERALL", **m_all})

results_df = pd.DataFrame(rows).set_index("field")
results_df

,TP,FP,FN,TN,accuracy,precision,recall,F1
field,,,,,,,,
full_name,66,0,5,22,0.9462,1.0000,0.9296,0.9635
session_identifier,84,0,2,7,0.9785,1.0000,0.9767,0.9882
gender,46,0,1,46,0.9892,1.0000,0.9787,0.9892
projectID_age,32,1,6,55,0.9255,0.9697,0.8421,0.9014
learning_route,47,1,1,44,0.9785,0.9792,0.9792,0.9792
participant_group,8,2,13,70,0.8387,0.8000,0.3810,0.5162
status,0,0,0,93,1.0000,NaN,NaN,NaN
place_of_origin,56,2,1,34,0.9677,0.9655,0.9825,0.9739
language_group,51,2,4,38,0.9368,0.9623,0.9273,0.9445
